# Task W1-02 — Strategy A: 추세 추종 일봉 백테스트

**Feature ID**: STR-A-001
**Sub-plan**: `docs/stage1-subplans/w1-02-strategy-a-daily.md`

Padysak/Vojtko 영감 추세 추종 전략을 일봉으로 백테스트.

## 사전 지정 파라미터 (변경 금지)

| 파라미터 | 값 | 설명 |
|---------|-----|------|
| MA_PERIOD | 200 | 장기 추세 필터 |
| DONCHIAN_HIGH | 20 | 돌파 신호 (20일 최고가) |
| DONCHIAN_LOW | 10 | 청산 신호 (10일 최저가) |
| VOL_AVG_PERIOD | 20 | 거래량 평균 윈도우 |
| VOL_MULT | 1.5 | 거래량 필터 배수 |
| SL_PCT | 0.08 | 하드 스톱 -8% (vectorbt sl_stop fraction) |

## 신호 로직

```
진입 (모두 충족):
  close > ma200 (대추세 상승)
  AND close > donchian_high.shift(1) (20일 최고가 돌파, look-ahead 차단)
  AND volume > vol_avg.shift(1) * 1.5 (거래량 동반)

청산 (하나 충족):
  close < donchian_low.shift(1) (10일 최저가 이탈)
  OR sl_stop=0.08 (-8% 하드 스톱, vectorbt 자동 처리)
```

## 검증된 vectorbt 0.28.5 API

- `sl_stop` fraction (0.08 = 8%)
- `sl_trail=False` (하드 스톱)
- `freq='1D'`
- `pf.sharpe_ratio()` 메서드 호출 (괄호 필수)
- ts_stop / td_stop / max_duration 사용 안 함


In [1]:
import pandas as pd
import numpy as np
import vectorbt as vbt
import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

from importlib.metadata import version
print(f"pandas:   {version('pandas')}")
print(f"numpy:    {version('numpy')}")
print(f"vectorbt: {version('vectorbt')}")
print(f"ta:       {version('ta')}")


pandas:   2.3.3
numpy:    2.3.5
vectorbt: 0.28.5
ta:       0.11.0


In [2]:
# 데이터 해시 검증 (consumer 노트북 첫 셀 룰, research/CLAUDE.md)
DATA_DIR = Path('../data')
PARQUET_NAME = 'KRW-BTC_1d_frozen_20260412.parquet'
PARQUET_PATH = DATA_DIR / PARQUET_NAME

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(65536), b''):
            h.update(chunk)
    return h.hexdigest()

# data_hashes.txt에서 expected 해시 읽기
with open(DATA_DIR / 'data_hashes.txt') as f:
    expected_hashes = {}
    for line in f:
        line = line.strip()
        if line and not line.startswith('#') and ':' in line:
            k, v = line.split(': ', 1)
            expected_hashes[k] = v

DATA_HASH = sha256_file(PARQUET_PATH)
expected_hash = expected_hashes[PARQUET_NAME]
assert DATA_HASH == expected_hash, f"Data hash mismatch! expected={expected_hash}, actual={DATA_HASH}"
print(f"Data hash verified: {DATA_HASH[:16]}...")


Data hash verified: da5b5a5bd74c1be0...


In [3]:
df = pd.read_parquet(PARQUET_PATH)
assert df.index.tz is not None and str(df.index.tz) == 'UTC', f"Expected UTC, got {df.index.tz}"

close  = df['close']
high   = df['high']
low    = df['low']
volume = df['volume']

print(f"Bars: {len(df)}")
print(f"Range: {df.index[0]} ~ {df.index[-1]}")
print(f"Columns: {list(df.columns)}")
print(f"NaN total: {df.isna().sum().sum()}")
df.head()


Bars: 1927
Range: 2021-01-01 00:00:00+00:00 ~ 2026-04-11 00:00:00+00:00
Columns: ['open', 'high', 'low', 'close', 'volume', 'value']
NaN total: 0


,open,high,low,close,volume,value
2021-01-01 00:00:00+00:00,32037000.0,32599000.0,31800000.0,32296000.0,5752.494216,1.856139e+11
2021-01-02 00:00:00+00:00,32295000.0,36600000.0,31920000.0,35700000.0,17451.167678,6.023355e+11
2021-01-03 00:00:00+00:00,35700000.0,39453000.0,35500000.0,37537000.0,25381.506623,9.588751e+11
2021-01-04 00:00:00+00:00,37537000.0,38476000.0,33000000.0,36460000.0,23598.194590,8.472770e+11
2021-01-05 00:00:00+00:00,36478000.0,38590000.0,34357000.0,38093000.0,13461.981681,4.915540e+11


In [4]:
# 사전 지정 파라미터 — 변경 금지 (Go/No-Go는 이 값으로만 평가)
MA_PERIOD       = 200
DONCHIAN_HIGH   = 20
DONCHIAN_LOW    = 10
VOL_AVG_PERIOD  = 20
VOL_MULT        = 1.5
SL_PCT          = 0.08    # vectorbt fraction (8%)

# 백테스트 설정
INIT_CASH = 1_000_000     # 100만원 (테스트용)
FEES      = 0.0005        # 업비트 0.05%
SLIPPAGE  = 0.0005        # 0.05% 슬리피지 추정
FREQ      = '1D'

# 주: ATR/Wilder 지표는 W1-04 강건성 분석에서 사용. W1-02는 고정 -8% 하드 스톱만.


In [5]:
# MA200 — 장기 추세 필터
ma200 = close.rolling(window=MA_PERIOD).mean()

# Donchian 채널 — .shift(1) 필수 (look-ahead 방지)
donchian_high = high.rolling(window=DONCHIAN_HIGH).max().shift(1)
donchian_low  = low.rolling(window=DONCHIAN_LOW).min().shift(1)

# 거래량 평균 — .shift(1) 필수 (신호 바의 평균은 알 수 없음)
vol_avg = volume.rolling(window=VOL_AVG_PERIOD).mean().shift(1)

# Sanity check
print(f"ma200 NaN (warmup): {ma200.isna().sum()} (expected: {MA_PERIOD - 1})")
print(f"donchian_high NaN: {donchian_high.isna().sum()} (expected: {DONCHIAN_HIGH})")
print(f"donchian_low NaN: {donchian_low.isna().sum()} (expected: {DONCHIAN_LOW})")
print(f"vol_avg NaN: {vol_avg.isna().sum()} (expected: {VOL_AVG_PERIOD})")


ma200 NaN (warmup): 199 (expected: 199)
donchian_high NaN: 20 (expected: 20)
donchian_low NaN: 10 (expected: 10)
vol_avg NaN: 20 (expected: 20)


In [6]:
# 진입 마스크 (Boolean Series)
entries = (close > ma200) & (close > donchian_high) & (volume > vol_avg * VOL_MULT)

# 청산 마스크 (Donchian low 이탈)
exits = close < donchian_low

# NaN을 False로 명시 (warmup 기간 안전)
entries = entries.fillna(False).astype(bool)
exits = exits.fillna(False).astype(bool)

print(f"Total entries: {entries.sum()}")
print(f"Total exits:   {exits.sum()}")

# Edge: 200 bar warmup 기간 신호 없음 확인
warmup_end = df.index[MA_PERIOD]
warmup_entries = entries.loc[:warmup_end].sum()
print(f"Warmup entries (first {MA_PERIOD} bars, ~{warmup_end.date()}): {warmup_entries} (expected: 0)")


Total entries: 46
Total exits:   123
Warmup entries (first 200 bars, ~2021-07-20): 0 (expected: 0)


In [7]:
# vectorbt 0.28.5 검증된 API만 사용
# - sl_stop: fraction (0.08 = 8%)
# - sl_trail: boolean (False = 하드 스톱)
# - freq: '1D' 일봉
# - ts_stop, td_stop, max_duration 사용 안 함

pf = vbt.Portfolio.from_signals(
    close=close,
    entries=entries,
    exits=exits,
    sl_stop=SL_PCT,
    sl_trail=False,
    init_cash=INIT_CASH,
    fees=FEES,
    slippage=SLIPPAGE,
    freq=FREQ,
)

print("vectorbt portfolio created successfully (no API errors)")


vectorbt portfolio created successfully (no API errors)


In [8]:
# 메서드 호출 (괄호 필수)
stats = pf.stats()
print(stats)


Start                          2021-01-01 00:00:00+00:00
End                            2026-04-11 00:00:00+00:00
Period                                1927 days 00:00:00
Start Value                                    1000000.0
End Value                                 2717593.259105
Total Return [%]                              171.759326
Benchmark Return [%]                          236.162373
Max Gross Exposure [%]                             100.0
Total Fees Paid                             27501.765919
Max Drawdown [%]                               22.450374
Max Drawdown Duration                  536 days 00:00:00
Total Trades                                          14
Total Closed Trades                                   14
Total Open Trades                                      0
Open Trade PnL                                       0.0
Win Rate [%]                                        50.0
Best Trade [%]                                 57.356969
Worst Trade [%]                

In [9]:
# 핵심 지표 추출 (모두 메서드 호출)
sharpe       = float(pf.sharpe_ratio())
total_return = float(pf.total_return())
max_dd       = float(pf.max_drawdown())
total_trades = int(pf.trades.count())

# Trades-related (있을 때만)
if total_trades > 0:
    win_rate      = float(pf.trades.win_rate())
    profit_factor = float(pf.trades.profit_factor())
else:
    win_rate = float('nan')
    profit_factor = float('nan')

print(f"Sharpe:        {sharpe:.4f}")
print(f"Total Return:  {total_return*100:.2f}%")
print(f"Max Drawdown:  {max_dd*100:.2f}%")
print(f"Win Rate:      {win_rate*100:.2f}%" if total_trades > 0 else "Win Rate:      N/A")
print(f"Profit Factor: {profit_factor:.3f}" if total_trades > 0 else "Profit Factor: N/A")
print(f"Total Trades:  {total_trades}")

# Week 1 Go 기준 평가 (W1-06에서 종합, 여기선 기록만)
print()
print("=== Week 1 Go 기준 평가 (참고) ===")
print(f"Sharpe > 0.8: {'PASS' if sharpe > 0.8 else 'FAIL'}")
print(f"MDD < 50%:    {'PASS' if abs(max_dd) < 0.5 else 'FAIL'}")


Sharpe:        1.0353
Total Return:  171.76%
Max Drawdown:  -22.45%
Win Rate:      50.00%
Profit Factor: 2.956
Total Trades:  14

=== Week 1 Go 기준 평가 (참고) ===
Sharpe > 0.8: PASS
MDD < 50%:    PASS


In [10]:
out_dir = Path('../outputs')
out_dir.mkdir(exist_ok=True)

results = {
    'feature_id': 'STR-A-001',
    'task_id': 'W1-02',
    'strategy': 'A_trend_daily',
    'description': 'Padysak/Vojtko-inspired trend-following: MA200 + Donchian(20/10) + Volume 1.5x + sl_stop 8%',
    'data_file': PARQUET_NAME,
    'data_hash': DATA_HASH,
    'data_bars': len(df),
    'data_range': [df.index[0].isoformat(), df.index[-1].isoformat()],
    'parameters': {
        'ma_period': MA_PERIOD,
        'donchian_high': DONCHIAN_HIGH,
        'donchian_low': DONCHIAN_LOW,
        'vol_avg_period': VOL_AVG_PERIOD,
        'vol_mult': VOL_MULT,
        'sl_pct': SL_PCT,
    },
    'backtest_config': {
        'init_cash': INIT_CASH,
        'fees': FEES,
        'slippage': SLIPPAGE,
        'freq': FREQ,
    },
    'metrics': {
        'sharpe': sharpe,
        'total_return': total_return,
        'max_drawdown': max_dd,
        'win_rate': win_rate if total_trades > 0 else None,
        'profit_factor': profit_factor if total_trades > 0 else None,
        'total_trades': total_trades,
    },
    'go_criteria_eval': {
        'sharpe_gt_0.8': sharpe > 0.8,
        'mdd_lt_50pct': abs(max_dd) < 0.5,
    },
    'generated_at': datetime.now(timezone.utc).isoformat(),
}

result_path = out_dir / 'strategy_a_daily.json'
with open(result_path, 'w') as f:
    json.dump(results, f, indent=2)

print(f"Saved: {result_path}")
print(json.dumps(results['metrics'], indent=2))


Saved: ../outputs/strategy_a_daily.json
{
  "sharpe": 1.0352900037639534,
  "total_return": 1.71759325910457,
  "max_drawdown": -0.22450373677773527,
  "win_rate": 0.5,
  "profit_factor": 2.9562104967125284,
  "total_trades": 14
}


In [11]:
import matplotlib
matplotlib.use('Agg')  # non-interactive backend (saves PNG without showing)
import matplotlib.pyplot as plt

equity = pf.value()
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True, gridspec_kw={'height_ratios': [3, 1]})

axes[0].plot(equity.index, equity.values, label='Strategy A equity', linewidth=1.2)
axes[0].axhline(INIT_CASH, color='gray', linestyle='--', alpha=0.5, label='Initial cash')
axes[0].set_title('Strategy A (Trend-Following Daily) — Equity Curve')
axes[0].set_ylabel('Portfolio Value (KRW)')
axes[0].legend(loc='upper left')
axes[0].grid(True, alpha=0.3)

drawdown = (equity / equity.cummax() - 1) * 100
axes[1].fill_between(drawdown.index, drawdown.values, 0, color='red', alpha=0.3)
axes[1].set_title('Drawdown (%)')
axes[1].set_ylabel('Drawdown %')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(out_dir / 'strategy_a_equity.png', dpi=100, bbox_inches='tight')
plt.close()
print(f"Equity curve saved: {out_dir / 'strategy_a_equity.png'}")


Matplotlib is building the font cache; this may take a moment.


Equity curve saved: ../outputs/strategy_a_equity.png


In [12]:
print("=" * 60)
print("W1-02 Strategy A Verification Summary")
print("=" * 60)
print(f"Feature ID:    STR-A-001")
print(f"Data:          {PARQUET_NAME}")
print(f"Data hash:     {DATA_HASH[:16]}...")
print(f"Bars:          {len(df)} (range {df.index[0].date()} ~ {df.index[-1].date()})")
print()
print("Pre-registered parameters:")
print(f"  MA={MA_PERIOD}, Donchian={DONCHIAN_HIGH}/{DONCHIAN_LOW}, Vol={VOL_AVG_PERIOD}*{VOL_MULT}, SL={SL_PCT}")
print()
print("Backtest results:")
print(f"  Sharpe:        {sharpe:.4f}")
print(f"  Total Return:  {total_return*100:.2f}%")
print(f"  Max Drawdown:  {max_dd*100:.2f}%")
print(f"  Total Trades:  {total_trades}")
if total_trades > 0:
    print(f"  Win Rate:      {win_rate*100:.2f}%")
    print(f"  Profit Factor: {profit_factor:.3f}")
print()
print("Go criteria (W1-06에서 종합 평가):")
print(f"  Sharpe > 0.8: {'PASS' if sharpe > 0.8 else 'FAIL'} ({sharpe:.4f})")
print(f"  MDD < 50%:    {'PASS' if abs(max_dd) < 0.5 else 'FAIL'} ({max_dd*100:.2f}%)")


W1-02 Strategy A Verification Summary
Feature ID:    STR-A-001
Data:          KRW-BTC_1d_frozen_20260412.parquet
Data hash:     da5b5a5bd74c1be0...
Bars:          1927 (range 2021-01-01 ~ 2026-04-11)

Pre-registered parameters:
  MA=200, Donchian=20/10, Vol=20*1.5, SL=0.08

Backtest results:
  Sharpe:        1.0353
  Total Return:  171.76%
  Max Drawdown:  -22.45%
  Total Trades:  14
  Win Rate:      50.00%
  Profit Factor: 2.956

Go criteria (W1-06에서 종합 평가):
  Sharpe > 0.8: PASS (1.0353)
  MDD < 50%:    PASS (-22.45%)
